


# Notebook 1 — static + risk trajectory + discharge retrieval encoder pipeline

This notebook does all expensive text processing and embedding work once, then saves the artifacts so the downstream classifier notebook can run without re-encoding.

## What this notebook does
1. Loads `ADMISSIONS.csv` and streams `NOTEEVENTS.csv`.
2. Uses early-note window logic using `MAX_HOSP_DAY`.
3. Builds:
   - admission-level static documents (`text_window`)
   - day-level documents (`text_day`) for risk trajectories
   - a scrubbed train-only discharge-summary retrieval bank
4. Encodes:
   - static admission documents
   - day-level documents
   - discharge-bank chunks
   - admission-query chunks
5. Computes admission-level retrieval features, including `wscore_top20`.
6. Saves everything needed for Notebook 2.

## Folder layout created on disk
Inside `EXP_DIR` this notebook creates:

- `metadata/`
  - split tables and lightweight metadata tables
  - experiment manifest / config
- `embeddings/`
  - `static_train.npz`, `static_val.npz`, `static_test.npz`
  - `day_train.npz`, `day_val.npz`, `day_test.npz` (only if `MAX_HOSP_DAY > 0`)
- `retrieval/`
  - bank/query metadata tables
  - bank/query chunk embeddings
  - admission-level retrieval features
  - optional diagnostics
- `models/`
  - left empty here; Notebook 2 (`02_train_evaluate_models`) saves downstream models there

## Important design choices used here
- The retrieval query text is the same admission text window used for the static embedding.
- Retrieval is computed for train / val / test.
- For train queries, retrieval excludes both:
  - same `HADM_ID`
  - same `SUBJECT_ID`
- The discharge bank is scrubbed with explicit outcome-term scrubber.
- The final downstream model in Notebook 2 will use:
  - static embedding
  - risk trajectory features
  - `wscore_top20`

## Note: as described in README, MIMIC-III is not included in this project, authorised PhysioNet access is required.

In [ ]:

# ============================================================
# CELL 1: Paths + experiment config
# - Edit this cell first if you want a different day window
# - Everything else uses these settings
# ============================================================

from pathlib import Path

# ----------------------------
# Core paths
# ----------------------------
ADMISSIONS_PATH = "your path to ADMISSIONS.csv"
NOTEEVENTS_PATH = "your path to NOTEEVENTS.csv"
MODEL_PATH = "your path to BiomedBERT-MIMIC-Contrastive"

# ----------------------------
# Experiment scope
# ----------------------------
MAX_HOSP_DAY = 2          # early-note window: keep hospital days 0..MAX_HOSP_DAY
DEV_SAMPLE = None         # set to an integer for fast debugging; None = full run
SEED = 42

# ----------------------------
# Note filtering
# ----------------------------
MIN_TEXT_LEN = 20
MAX_NOTES_PER_HADM = 60   # cap early non-discharge notes per admission

# ----------------------------
# Streaming / performance
# ----------------------------
CHUNKSIZE = 100_000       # NOTEEVENTS streaming chunk size

# ----------------------------
# Static document encoding
# These are currently optimized for 90GB of VRAM.
# Be wary of OOM
# ----------------------------
STATIC_BATCH_SIZE = 6144
STATIC_CHUNK_CHARS = 2000
STATIC_OVERLAP = 200
STATIC_MAX_CHUNKS = 16

# ----------------------------
# Day document encoding
# Same as above
# ----------------------------
DAY_BATCH_SIZE = 6144
DAY_CHUNK_CHARS = 2000
DAY_OVERLAP = 200
DAY_MAX_CHUNKS = 12

# ----------------------------
# Retrieval chunking
# Word chunking is kept separate from above's char chunking because
# retrieval works best on smaller bank/query chunks.
# ----------------------------
BANK_CHUNK_WORDS = 220
BANK_CHUNK_OVERLAP = 40

QUERY_CHUNK_WORDS = 180
QUERY_CHUNK_OVERLAP = 40

# ----------------------------
# Retrieval settings
# ----------------------------
TOP_K_LIST = [20, 50]
MAX_FINAL_K = max(TOP_K_LIST)
RETRIEVAL_PER_QUERY_CHUNK = 100
RETR_BLOCK_SIZE = 2048
RETR_BATCH_SIZE = 512

# ----------------------------
# Save / cache behaviour
# If False, existing saved artifacts are reused instead of recomputed.
# ----------------------------
FORCE_REBUILD_STATIC = False
FORCE_REBUILD_DAY = False
FORCE_REBUILD_RETRIEVAL = False

# Save chunk text metadata too?
# True keeps every encoder-side retrieval artifact available for reuse.
SAVE_CHUNK_TEXT_METADATA = True

# ----------------------------
# Output folders
# ----------------------------
EXPERIMENT_NAME = f"your experiment name"
ARTIFACT_ROOT = Path("your artifact root")
EXP_DIR = ARTIFACT_ROOT / EXPERIMENT_NAME

META_DIR = EXP_DIR / "metadata"
EMB_DIR = EXP_DIR / "embeddings"
RETR_DIR = EXP_DIR / "retrieval"
MODEL_DIR = EXP_DIR / "models"

print("Experiment directory:")
print(EXP_DIR)


In [ ]:

# ============================================================
# CELL 2: Imports + reproducibility
# - Core Python / data handling
# - SentenceTransformer for embeddings
# - Torch for GPU retrieval
# ============================================================

import gc
import json
import math
import os
import pickle
import random
import re
from typing import Dict, List, Tuple

import numpy as np
import pandas as pd
import torch
from sentence_transformers import SentenceTransformer
from tqdm.auto import tqdm
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, average_precision_score

from IPython.display import display

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# Reproducibility
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# Helpful GPU settings on Colab
if DEVICE == "cuda":
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    try:
        torch.set_float32_matmul_precision("high")
    except Exception:
        pass

print("DEVICE:", DEVICE)


In [ ]:

# ============================================================
# CELL 3: Helper functions
# - File helpers
# - Outcome scrubber
# - Document chunking + pooling
# - Retrieval chunking + scoring
# ============================================================

def ensure_dir(path: Path):
    path.mkdir(parents=True, exist_ok=True)

for folder in [META_DIR, EMB_DIR, RETR_DIR, MODEL_DIR]:
    ensure_dir(folder)


def save_npz_array(path: Path, array: np.ndarray):
    path.parent.mkdir(parents=True, exist_ok=True)
    np.savez_compressed(path, array=array)


def load_npz_array(path: Path) -> np.ndarray:
    return np.load(path)["array"]


def save_pickle(obj, path: Path):
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "wb") as f:
        pickle.dump(obj, f)


def load_pickle(path: Path):
    with open(path, "rb") as f:
        return pickle.load(f)


def save_table(df: pd.DataFrame, path: Path):
    path.parent.mkdir(parents=True, exist_ok=True)
    df.to_parquet(path, index=False)


def safe_to_datetime(series: pd.Series) -> pd.Series:
    return pd.to_datetime(series, errors="coerce")


# ---------- Explicit outcome leakage scrubber ----------
EXPLICIT_OUTCOME_TERMS = [
    r"\bdeath\b",
    r"\bdied\b",
    r"\bexpired\b",
    r"\bdnr\b",
    r"\bdni\b",
    r"\bcmo\b",
    r"\bcomfort measures\b",
    r"\bcomfort care\b",
    r"\bcardiac arrest\b",
    r"\barrest\b",
    r"\bcpr\b",
    r"\bpost mortem\b",
]
_OUTCOME_PAT = re.compile("|".join(EXPLICIT_OUTCOME_TERMS), flags=re.IGNORECASE)

def scrub_outcome_terms(text_series: pd.Series) -> pd.Series:
    """
    Removes explicit label-leaking terms before any embeddings are created.
    """
    return text_series.astype(str).str.replace(_OUTCOME_PAT, " ", regex=True)


def normalize_text_for_retrieval(x: str) -> str:
    """
    Light text normalisation used on retrieval chunk text.
    We keep this minimal because the major leakage scrubber already ran.
    """
    if pd.isna(x):
        return ""
    x = str(x).replace("\r", " ").replace("\n", " ")
    x = " ".join(x.split())
    return x.strip()


# ---------- ComboV6-style long-document chunking ----------
def chunk_text_spread(text: str, chunk_chars: int, overlap: int, max_chunks: int) -> List[str]:
    """
    Splits one long document into overlapping character chunks.
    If there are too many chunks, sample them evenly across the document.
    """
    text = str(text)
    if not text.strip():
        return []

    step = max(1, chunk_chars - overlap)
    chunks = [text[i:i + chunk_chars] for i in range(0, len(text), step)]

    if len(chunks) <= max_chunks:
        return chunks

    idx = np.linspace(0, len(chunks) - 1, max_chunks).astype(int)
    return [chunks[i] for i in idx]


def encode_docs_fast(
    texts,
    model: SentenceTransformer,
    batch_size: int,
    chunk_chars: int,
    overlap: int,
    max_chunks: int,
    pooling: str = "mean_max",
    normalize_after: bool = True
) -> np.ndarray:
    """
    ComboV6-style efficient encoder:
    1) chunk each document
    2) encode all chunks in big GPU batches
    3) pool back to one vector per original document
    """
    texts = list(texts)
    all_chunks = []
    spans = []

    for t in tqdm(texts, desc="Chunking long documents"):
        chunks = chunk_text_spread(
            t,
            chunk_chars=chunk_chars,
            overlap=overlap,
            max_chunks=max_chunks
        )
        start = len(all_chunks)
        all_chunks.extend(chunks)
        end = len(all_chunks)
        spans.append((start, end))

    print("Total chunks to encode:", len(all_chunks))

    if len(all_chunks) == 0:
        dim = model.get_sentence_embedding_dimension()
        out_dim = dim if pooling in ("mean", "max") else 2 * dim
        return np.zeros((len(texts), out_dim), dtype=np.float32)

    chunk_embs = model.encode(
        all_chunks,
        batch_size=batch_size,
        convert_to_numpy=True,
        normalize_embeddings=True,
        show_progress_bar=True,
    ).astype(np.float32)

    dim = chunk_embs.shape[1]
    out_dim = dim if pooling in ("mean", "max") else 2 * dim
    X = np.zeros((len(spans), out_dim), dtype=np.float32)

    for i, (s, e) in enumerate(spans):
        if s == e:
            continue

        embs = chunk_embs[s:e]

        if pooling == "mean":
            v = embs.mean(axis=0)
        elif pooling == "max":
            v = embs.max(axis=0)
        elif pooling == "mean_max":
            v = np.concatenate([embs.mean(axis=0), embs.max(axis=0)])
        else:
            raise ValueError("pooling must be 'mean', 'max', or 'mean_max'")

        if normalize_after:
            v = v / (np.linalg.norm(v) + 1e-12)

        X[i] = v.astype(np.float32)

    return X


# ---------- Retrieval helpers ----------
def chunk_text_by_words(text: str, chunk_words: int, overlap_words: int) -> List[str]:
    """
    Splits text into overlapping word chunks.
    Example: chunk_words=200, overlap_words=40 -> next chunk starts 160 words later.
    """
    text = normalize_text_for_retrieval(text)
    if not text:
        return []

    words = text.split()
    if len(words) <= chunk_words:
        return [" ".join(words)]

    step = max(1, chunk_words - overlap_words)
    chunks = []

    for start in range(0, len(words), step):
        end = start + chunk_words
        block = words[start:end]
        if len(block) == 0:
            continue
        chunks.append(" ".join(block))
        if end >= len(words):
            break

    return chunks


def l2_normalize_np(x: np.ndarray) -> np.ndarray:
    norms = np.linalg.norm(x, axis=1, keepdims=True)
    norms = np.clip(norms, 1e-12, None)
    return x / norms


def embed_texts(texts: List[str], model: SentenceTransformer, batch_size: int) -> np.ndarray:
    """
    Encodes a list of short chunk texts for retrieval and returns L2-normalised vectors.
    """
    emb = model.encode(
        texts,
        batch_size=batch_size,
        convert_to_numpy=True,
        normalize_embeddings=True,
        show_progress_bar=True,
    ).astype(np.float32)
    return l2_normalize_np(emb)


def weighted_neighbor_score(similarities: np.ndarray, labels: np.ndarray) -> float:
    """
    Similarity-weighted fraction of positive neighbours.
    Similarities are clipped at 0 so negative cosine values do not create odd weights.
    """
    if len(labels) == 0:
        return np.nan

    sims = np.maximum(similarities.astype(float), 0.0)
    labs = labels.astype(float)

    if sims.sum() <= 1e-12:
        return float(labs.mean())

    return float((sims * labs).sum() / sims.sum())


def metrics_binary(y_true: np.ndarray, y_score: np.ndarray) -> Dict[str, float]:
    """
    AUROC and AUPRC for binary labels.
    Returns NaN if only one class is present.
    """
    if len(np.unique(y_true)) < 2:
        return {"AUROC": np.nan, "AUPRC": np.nan}

    return {
        "AUROC": float(roc_auc_score(y_true, y_score)),
        "AUPRC": float(average_precision_score(y_true, y_score)),
    }


def retrieve_topk_torch_gpu(
    query_emb: np.ndarray,
    bank_emb: np.ndarray,
    topk: int = 100,
    block_size: int = 2048,
    use_fp16: bool = True,
):
    """
    Exact retrieval using torch matmul on GPU.
    Query and bank vectors are assumed to already be L2-normalised.
    """
    device = "cuda" if torch.cuda.is_available() else "cpu"
    topk = min(topk, bank_emb.shape[0])

    dtype = torch.float16 if (use_fp16 and device == "cuda") else torch.float32

    bank_t = torch.from_numpy(bank_emb).to(device=device, dtype=dtype)
    query_t = torch.from_numpy(query_emb).to(device=device, dtype=dtype)

    all_idx = []
    all_sim = []

    with torch.no_grad():
        for start in tqdm(range(0, query_t.shape[0], block_size), desc="Retrieving query chunks"):
            end = min(start + block_size, query_t.shape[0])
            sims = query_t[start:end] @ bank_t.T
            sim_vals, idx_vals = torch.topk(sims, k=topk, dim=1)

            all_idx.append(idx_vals.cpu().numpy().astype(np.int32))
            all_sim.append(sim_vals.cpu().numpy().astype(np.float32))

            del sims, sim_vals, idx_vals
            if device == "cuda":
                torch.cuda.empty_cache()

    return np.vstack(all_idx), np.vstack(all_sim)


In [ ]:

# ============================================================
# CELL 4: Load ADMISSIONS
# - We keep ADMITTIME because early-note day indexing uses it
# ============================================================

adm = pd.read_csv(
    ADMISSIONS_PATH,
    usecols=["SUBJECT_ID", "HADM_ID", "HOSPITAL_EXPIRE_FLAG", "ADMITTIME"],
    parse_dates=["ADMITTIME"],
)

adm = adm.dropna(subset=["SUBJECT_ID", "HADM_ID", "ADMITTIME"]).copy()
adm["SUBJECT_ID"] = adm["SUBJECT_ID"].astype(int)
adm["HADM_ID"] = adm["HADM_ID"].astype(int)
adm["label"] = adm["HOSPITAL_EXPIRE_FLAG"].fillna(0).astype(int)
adm["admit_date"] = adm["ADMITTIME"].dt.normalize()

adm = adm[["SUBJECT_ID", "HADM_ID", "ADMITTIME", "admit_date", "label"]].copy()

print("ADMISSIONS shape:", adm.shape)
print("Mortality prevalence:", adm["label"].mean())
display(adm.head())


In [ ]:

# ============================================================
# CELL 5: Stream NOTEEVENTS once
# - Builds two note collections in one pass:
#   1) early non-discharge notes for static/day pipelines
#   2) discharge summaries for the retrieval bank
# - The same explicit scrubber is applied to both
# ============================================================

adm_lookup = adm.set_index("HADM_ID")[["SUBJECT_ID", "ADMITTIME", "admit_date", "label"]]

early_rows = []
discharge_rows = []
total_note_rows_seen = 0

usecols_notes = ["ROW_ID", "HADM_ID", "CHARTDATE", "CHARTTIME", "CATEGORY", "TEXT"]

for chunk in pd.read_csv(
    NOTEEVENTS_PATH,
    usecols=usecols_notes,
    chunksize=CHUNKSIZE,
    low_memory=False
):
    total_note_rows_seen += len(chunk)

    chunk = chunk.dropna(subset=["HADM_ID", "TEXT"]).copy()
    if chunk.empty:
        continue

    chunk["HADM_ID"] = pd.to_numeric(chunk["HADM_ID"], errors="coerce")
    chunk = chunk.dropna(subset=["HADM_ID"]).copy()
    chunk["HADM_ID"] = chunk["HADM_ID"].astype(int)

    # Keep only admissions in our cohort
    chunk = chunk[chunk["HADM_ID"].isin(adm_lookup.index)]
    if chunk.empty:
        continue

    chunk["CATEGORY"] = chunk["CATEGORY"].fillna("").astype(str)
    chunk["TEXT"] = chunk["TEXT"].astype(str)
    chunk = chunk[chunk["TEXT"].str.len() >= MIN_TEXT_LEN].copy()
    if chunk.empty:
        continue

    # Parse times used for day indexing and discharge ordering
    chunk["note_date"] = pd.to_datetime(chunk["CHARTDATE"], errors="coerce").dt.normalize()
    chunk["note_time"] = pd.to_datetime(chunk["CHARTTIME"], errors="coerce")
    chunk["note_time"] = chunk["note_time"].fillna(pd.to_datetime(chunk["CHARTDATE"], errors="coerce"))

    # Join admission metadata
    chunk = chunk.join(adm_lookup, on="HADM_ID", how="inner")
    if chunk.empty:
        continue

    chunk["is_discharge"] = chunk["CATEGORY"].str.lower().eq("discharge summary")

    # -------------------------
    # Early non-discharge notes
    # -------------------------
    early = chunk[~chunk["is_discharge"]].copy()
    if not early.empty:
        early = early.dropna(subset=["note_date"]).copy()
        early["hospital_day"] = (early["note_date"] - early["admit_date"]).dt.days
        early = early[(early["hospital_day"] >= 0) & (early["hospital_day"] <= MAX_HOSP_DAY)].copy()

        if not early.empty:
            early["text_clean"] = scrub_outcome_terms(early["TEXT"])
            early["text_clean"] = early["text_clean"].astype(str).str.strip()
            early = early[early["text_clean"].str.len() > 0].copy()

            if not early.empty:
                early_rows.append(
                    early[[
                        "SUBJECT_ID", "HADM_ID", "admit_date", "note_date",
                        "hospital_day", "text_clean", "label"
                    ]].copy()
                )

    # -------------------------
    # Discharge summaries (retrieval bank path)
    # We do NOT restrict these by hospital day.
    # -------------------------
    discharge = chunk[chunk["is_discharge"]].copy()
    if not discharge.empty:
        discharge["text_clean"] = scrub_outcome_terms(discharge["TEXT"])
        discharge["text_clean"] = discharge["text_clean"].astype(str).str.strip()
        discharge = discharge[discharge["text_clean"].str.len() > 0].copy()

        if not discharge.empty:
            discharge_rows.append(
                discharge[[
                    "ROW_ID", "SUBJECT_ID", "HADM_ID", "note_time",
                    "text_clean", "label"
                ]].copy()
            )

notes_df = pd.concat(early_rows, ignore_index=True) if len(early_rows) else pd.DataFrame(
    columns=["SUBJECT_ID", "HADM_ID", "admit_date", "note_date", "hospital_day", "text_clean", "label"]
)

discharge_notes_df = pd.concat(discharge_rows, ignore_index=True) if len(discharge_rows) else pd.DataFrame(
    columns=["ROW_ID", "SUBJECT_ID", "HADM_ID", "note_time", "text_clean", "label"]
)

notes_df["SUBJECT_ID"] = pd.to_numeric(notes_df["SUBJECT_ID"], errors="coerce").fillna(-1).astype(int)
notes_df["HADM_ID"] = pd.to_numeric(notes_df["HADM_ID"], errors="coerce").fillna(-1).astype(int)
notes_df["label"] = pd.to_numeric(notes_df["label"], errors="coerce").fillna(0).astype(int)

if len(discharge_notes_df):
    discharge_notes_df["SUBJECT_ID"] = pd.to_numeric(discharge_notes_df["SUBJECT_ID"], errors="coerce").fillna(-1).astype(int)
    discharge_notes_df["HADM_ID"] = pd.to_numeric(discharge_notes_df["HADM_ID"], errors="coerce").fillna(-1).astype(int)
    discharge_notes_df["label"] = pd.to_numeric(discharge_notes_df["label"], errors="coerce").fillna(0).astype(int)

print("Total streamed NOTEEVENTS rows seen:", total_note_rows_seen)
print("Early non-discharge notes kept:", notes_df.shape)
print("Discharge note rows kept:", discharge_notes_df.shape)
display(notes_df.head())
display(discharge_notes_df.head())


In [ ]:

# ============================================================
# CELL 6: Build admission-level docs, day-level docs, and discharge corpus
# - admission_docs -> static embedding input
# - day_docs       -> risk trajectory input
# - discharge_corpus -> train-only bank source after split assignment
# ============================================================

# ---- Sort + cap early notes per admission to keep runtime manageable ----
notes_df = notes_df.sort_values(["HADM_ID", "hospital_day", "note_date"]).reset_index(drop=True)

notes_df = (
    notes_df.groupby("HADM_ID", group_keys=False)
            .head(MAX_NOTES_PER_HADM)
            .reset_index(drop=True)
)

print("Early notes after per-admission cap:", notes_df.shape)

# ---- Admission-level static text window ----
admission_docs = (
    notes_df.groupby(["HADM_ID", "SUBJECT_ID"], as_index=False)
            .agg(
                admit_date=("admit_date", "first"),
                label=("label", "first"),
                text_window=("text_clean", lambda s: "\n\n----- NOTE -----\n\n".join(s.dropna()))
            )
)

admission_docs["text_window_len"] = admission_docs["text_window"].astype(str).str.len()

# ---- Optional dev sampling ----
# Apply this once at the admission level so static, day, and retrieval stay aligned.
if DEV_SAMPLE is not None:
    if DEV_SAMPLE > len(admission_docs):
        raise ValueError(f"DEV_SAMPLE={DEV_SAMPLE} is larger than available admissions {len(admission_docs)}")

    admission_docs = admission_docs.sample(DEV_SAMPLE, random_state=SEED).reset_index(drop=True)
    sampled_hadm_ids = set(admission_docs["HADM_ID"].tolist())

    notes_df = notes_df[notes_df["HADM_ID"].isin(sampled_hadm_ids)].reset_index(drop=True)
    discharge_notes_df = discharge_notes_df[discharge_notes_df["HADM_ID"].isin(sampled_hadm_ids)].reset_index(drop=True)

    print("After DEV_SAMPLE:")
    print("admission_docs:", admission_docs.shape)
    print("notes_df:", notes_df.shape)
    print("discharge_notes_df:", discharge_notes_df.shape)

# ---- Day-level docs (used later by clf_static -> risk trajectory) ----
if MAX_HOSP_DAY == 0:
    day_docs = pd.DataFrame(columns=["HADM_ID", "hospital_day", "SUBJECT_ID", "label", "text_day", "text_day_len"])
else:
    day_docs = (
        notes_df.groupby(["HADM_ID", "hospital_day"], as_index=False)
                .agg(
                    SUBJECT_ID=("SUBJECT_ID", "first"),
                    label=("label", "first"),
                    text_day=("text_clean", lambda s: "\n\n----- NOTE -----\n\n".join(s.dropna()))
                )
    )
    day_docs["text_day_len"] = day_docs["text_day"].astype(str).str.len()

# ---- Discharge summaries collapsed to one document per admission ----
if len(discharge_notes_df) == 0:
    discharge_corpus = pd.DataFrame(columns=["HADM_ID", "SUBJECT_ID", "label", "discharge_text", "discharge_text_len"])
else:
    discharge_notes_df = discharge_notes_df.sort_values(["HADM_ID", "note_time", "ROW_ID"], na_position="last").reset_index(drop=True)

    discharge_corpus = (
        discharge_notes_df.groupby(["HADM_ID", "SUBJECT_ID"], as_index=False)
                          .agg(
                              label=("label", "first"),
                              discharge_text=("text_clean", lambda s: "\n\n".join([t for t in s if isinstance(t, str) and len(t) > 0]))
                          )
    )
    discharge_corpus["discharge_text_len"] = discharge_corpus["discharge_text"].astype(str).str.len()
    discharge_corpus = discharge_corpus[discharge_corpus["discharge_text_len"] > 0].reset_index(drop=True)

print("Admissions in early-note cohort:", admission_docs.shape)
print("Day docs:", day_docs.shape)
print("Discharge corpus:", discharge_corpus.shape)

display(admission_docs.head())
display(day_docs.head())
display(discharge_corpus.head())


In [ ]:

# ============================================================
# CELL 7: Subject-level train / val / test splits
# - Split by SUBJECT_ID to avoid patient leakage
# - Attach split labels to static/day/retrieval tables
# ============================================================

subjects = admission_docs["SUBJECT_ID"].drop_duplicates().values
train_subj, temp_subj = train_test_split(subjects, test_size=0.30, random_state=SEED)
val_subj, test_subj = train_test_split(temp_subj, test_size=0.50, random_state=SEED)

split_map = {}
for s in train_subj:
    split_map[int(s)] = "train"
for s in val_subj:
    split_map[int(s)] = "val"
for s in test_subj:
    split_map[int(s)] = "test"

admission_docs["split"] = admission_docs["SUBJECT_ID"].map(split_map)

if len(day_docs):
    day_docs["split"] = day_docs["SUBJECT_ID"].map(split_map)

if len(discharge_corpus):
    discharge_corpus["split"] = discharge_corpus["SUBJECT_ID"].map(split_map)

# Split-specific admission tables
train_adm = admission_docs[admission_docs["split"] == "train"].reset_index(drop=True)
val_adm   = admission_docs[admission_docs["split"] == "val"].reset_index(drop=True)
test_adm  = admission_docs[admission_docs["split"] == "test"].reset_index(drop=True)

# Split-specific day tables
if MAX_HOSP_DAY == 0:
    train_day = pd.DataFrame(columns=day_docs.columns)
    val_day = pd.DataFrame(columns=day_docs.columns)
    test_day = pd.DataFrame(columns=day_docs.columns)
else:
    train_day = day_docs[day_docs["split"] == "train"].reset_index(drop=True)
    val_day   = day_docs[day_docs["split"] == "val"].reset_index(drop=True)
    test_day  = day_docs[day_docs["split"] == "test"].reset_index(drop=True)

# Train-only discharge bank
bank_df = discharge_corpus[discharge_corpus["split"] == "train"].reset_index(drop=True)

# Admission-level query documents for retrieval
# Query text exactly matches the same early-note window used for the static embedding.
query_docs = admission_docs[["HADM_ID", "SUBJECT_ID", "split", "label", "text_window"]].copy()
query_docs = query_docs.rename(columns={"text_window": "query_text"})
query_docs["query_text_len"] = query_docs["query_text"].astype(str).str.len()
query_docs = query_docs[query_docs["query_text_len"] > 0].reset_index(drop=True)
query_docs["query_id"] = query_docs.index.astype(int)


if len(bank_df) == 0:
    raise ValueError("The train discharge-summary bank is empty. Retrieval cannot run without train discharge summaries.")

if len(query_docs) == 0:
    raise ValueError("No admission-level query documents were built. Check the early-note filtering window.")

print("Static split sizes:", train_adm.shape, val_adm.shape, test_adm.shape)
print("Day split sizes:", train_day.shape, val_day.shape, test_day.shape)
print("Train discharge bank:", bank_df.shape)
print("Query docs:", query_docs.shape)

print("\nMortality prevalence by split:")
print(admission_docs.groupby("split")["label"].mean())

display(query_docs.head())
display(bank_df.head())


In [ ]:

# ============================================================
# CELL 8: Save lightweight metadata tables now
# - These tables are all Notebook 2 needs for alignment and labels
# - We do NOT save full raw note text by default because that would
#   consume a lot of Drive storage
# ============================================================

# Admission metadata tables
save_table(
    admission_docs[["HADM_ID", "SUBJECT_ID", "admit_date", "label", "split", "text_window_len"]].copy(),
    META_DIR / "admission_docs_meta.parquet"
)
save_table(
    train_adm[["HADM_ID", "SUBJECT_ID", "admit_date", "label", "split", "text_window_len"]].copy(),
    META_DIR / "train_adm_meta.parquet"
)
save_table(
    val_adm[["HADM_ID", "SUBJECT_ID", "admit_date", "label", "split", "text_window_len"]].copy(),
    META_DIR / "val_adm_meta.parquet"
)
save_table(
    test_adm[["HADM_ID", "SUBJECT_ID", "admit_date", "label", "split", "text_window_len"]].copy(),
    META_DIR / "test_adm_meta.parquet"
)

# Day metadata tables
if MAX_HOSP_DAY > 0:
    save_table(
        train_day[["HADM_ID", "hospital_day", "SUBJECT_ID", "label", "split", "text_day_len"]].copy(),
        META_DIR / "train_day_meta.parquet"
    )
    save_table(
        val_day[["HADM_ID", "hospital_day", "SUBJECT_ID", "label", "split", "text_day_len"]].copy(),
        META_DIR / "val_day_meta.parquet"
    )
    save_table(
        test_day[["HADM_ID", "hospital_day", "SUBJECT_ID", "label", "split", "text_day_len"]].copy(),
        META_DIR / "test_day_meta.parquet"
    )

# Retrieval metadata tables
save_table(
    bank_df[["HADM_ID", "SUBJECT_ID", "label", "split", "discharge_text_len"]].copy(),
    RETR_DIR / "bank_df_meta.parquet"
)
save_table(
    query_docs[["query_id", "HADM_ID", "SUBJECT_ID", "split", "label", "query_text_len"]].copy(),
    RETR_DIR / "query_docs_meta.parquet"
)

# Save a simple manifest so Notebook 2 can sanity-check what was built.
manifest = {
    "experiment_name": EXPERIMENT_NAME,
    "max_hosp_day": int(MAX_HOSP_DAY),
    "seed": int(SEED),
    "model_path": MODEL_PATH,
    "static_batch_size": int(STATIC_BATCH_SIZE),
    "day_batch_size": int(DAY_BATCH_SIZE),
    "retr_batch_size": int(RETR_BATCH_SIZE),
    "top_k_list": TOP_K_LIST,
    "max_final_k": int(MAX_FINAL_K),
    "notes_cap_per_hadm": int(MAX_NOTES_PER_HADM),
    "dev_sample": None if DEV_SAMPLE is None else int(DEV_SAMPLE),
}

with open(META_DIR / "manifest.json", "w") as f:
    json.dump(manifest, f, indent=2)

print("Metadata + manifest saved.")


In [ ]:

# ============================================================
# CELL 9: Load the encoder
# - One shared encoder is used for:
#   static docs
#   day docs
#   retrieval bank chunks
#   retrieval query chunks
# ============================================================

print("Loading encoder from:", MODEL_PATH)
encoder = SentenceTransformer(MODEL_PATH, device=DEVICE)

# Using fp16 on CUDA helps throughput and memory usage for inference.
if DEVICE == "cuda":
    try:
        encoder.half()
        print("Encoder switched to fp16 for inference.")
    except Exception as e:
        print("Could not switch encoder to fp16:", e)

print("Sentence embedding dimension:", encoder.get_sentence_embedding_dimension())


In [ ]:

# ============================================================
# CELL 10: Encode static admission docs (with caching)
# - These are the main admission-level embeddings
# - Notebook 2 will load these directly instead of re-encoding
# ============================================================

static_train_path = EMB_DIR / "static_train.npz"
static_val_path   = EMB_DIR / "static_val.npz"
static_test_path  = EMB_DIR / "static_test.npz"

if (
    static_train_path.exists() and static_val_path.exists() and static_test_path.exists()
    and not FORCE_REBUILD_STATIC
):
    print("Loading cached static embeddings from Drive...")
    X_train_static = load_npz_array(static_train_path)
    X_val_static   = load_npz_array(static_val_path)
    X_test_static  = load_npz_array(static_test_path)
else:
    print("Encoding static admission documents...")
    X_train_static = encode_docs_fast(
        train_adm["text_window"],
        encoder,
        batch_size=STATIC_BATCH_SIZE,
        chunk_chars=STATIC_CHUNK_CHARS,
        overlap=STATIC_OVERLAP,
        max_chunks=STATIC_MAX_CHUNKS,
        pooling="mean_max",
    )
    X_val_static = encode_docs_fast(
        val_adm["text_window"],
        encoder,
        batch_size=STATIC_BATCH_SIZE,
        chunk_chars=STATIC_CHUNK_CHARS,
        overlap=STATIC_OVERLAP,
        max_chunks=STATIC_MAX_CHUNKS,
        pooling="mean_max",
    )
    X_test_static = encode_docs_fast(
        test_adm["text_window"],
        encoder,
        batch_size=STATIC_BATCH_SIZE,
        chunk_chars=STATIC_CHUNK_CHARS,
        overlap=STATIC_OVERLAP,
        max_chunks=STATIC_MAX_CHUNKS,
        pooling="mean_max",
    )

    save_npz_array(static_train_path, X_train_static)
    save_npz_array(static_val_path, X_val_static)
    save_npz_array(static_test_path, X_test_static)

print("X_train_static:", X_train_static.shape)
print("X_val_static:", X_val_static.shape)
print("X_test_static:", X_test_static.shape)


In [ ]:

# ============================================================
# CELL 11: Encode day docs (with caching)
# - Only needed when MAX_HOSP_DAY > 0
# - Notebook 2 uses these with clf_static to build risk trajectory features
# ============================================================

if MAX_HOSP_DAY == 0:
    print("MAX_HOSP_DAY == 0 -> no day embeddings are needed.")
else:
    day_train_path = EMB_DIR / "day_train.npz"
    day_val_path   = EMB_DIR / "day_val.npz"
    day_test_path  = EMB_DIR / "day_test.npz"

    if (
        day_train_path.exists() and day_val_path.exists() and day_test_path.exists()
        and not FORCE_REBUILD_DAY
    ):
        print("Loading cached day embeddings from Drive...")
        X_train_day = load_npz_array(day_train_path)
        X_val_day   = load_npz_array(day_val_path)
        X_test_day  = load_npz_array(day_test_path)
    else:
        print("Encoding day-level documents...")
        X_train_day = encode_docs_fast(
            train_day["text_day"],
            encoder,
            batch_size=DAY_BATCH_SIZE,
            chunk_chars=DAY_CHUNK_CHARS,
            overlap=DAY_OVERLAP,
            max_chunks=DAY_MAX_CHUNKS,
            pooling="mean_max",
        )
        X_val_day = encode_docs_fast(
            val_day["text_day"],
            encoder,
            batch_size=DAY_BATCH_SIZE,
            chunk_chars=DAY_CHUNK_CHARS,
            overlap=DAY_OVERLAP,
            max_chunks=DAY_MAX_CHUNKS,
            pooling="mean_max",
        )
        X_test_day = encode_docs_fast(
            test_day["text_day"],
            encoder,
            batch_size=DAY_BATCH_SIZE,
            chunk_chars=DAY_CHUNK_CHARS,
            overlap=DAY_OVERLAP,
            max_chunks=DAY_MAX_CHUNKS,
            pooling="mean_max",
        )

        save_npz_array(day_train_path, X_train_day)
        save_npz_array(day_val_path, X_val_day)
        save_npz_array(day_test_path, X_test_day)

    print("X_train_day:", X_train_day.shape)
    print("X_val_day:", X_val_day.shape)
    print("X_test_day:", X_test_day.shape)


In [ ]:

# ============================================================
# CELL 12: Build retrieval bank/query chunks
# - Bank = scrubbed train discharge summaries
# - Query = admission-level early-note text window
# - Query text matches the static embedding window exactly
# ============================================================

bank_chunks_path = RETR_DIR / "bank_chunks_meta.parquet"
query_chunks_path = RETR_DIR / "query_chunks_meta.parquet"

need_rebuild_chunk_tables = (
    FORCE_REBUILD_RETRIEVAL
    or (not bank_chunks_path.exists())
    or (not query_chunks_path.exists())
)

if not need_rebuild_chunk_tables:
    print("Loading cached retrieval chunk metadata...")
    bank_chunks_df = pd.read_parquet(bank_chunks_path)
    query_chunks_df = pd.read_parquet(query_chunks_path)
else:
    print("Building retrieval bank chunks...")
    bank_chunk_rows = []

    for _, row in tqdm(bank_df.iterrows(), total=len(bank_df), desc="Chunking bank documents"):
        chunks = chunk_text_by_words(
            row["discharge_text"],
            chunk_words=BANK_CHUNK_WORDS,
            overlap_words=BANK_CHUNK_OVERLAP
        )

        for chunk_rank, chunk_text in enumerate(chunks):
            bank_chunk_rows.append({
                "HADM_ID": int(row["HADM_ID"]),
                "SUBJECT_ID": int(row["SUBJECT_ID"]),
                "split": row["split"],
                "label": int(row["label"]),
                "chunk_rank": int(chunk_rank),
                "chunk_text": chunk_text,
                "chunk_text_len": len(chunk_text),
                "chunk_word_len": len(chunk_text.split()),
            })

    bank_chunks_df = pd.DataFrame(bank_chunk_rows).reset_index(drop=True)
    bank_chunks_df["bank_chunk_idx"] = bank_chunks_df.index.astype(int)

    print("Building retrieval query chunks...")
    query_chunk_rows = []

    for _, row in tqdm(query_docs.iterrows(), total=len(query_docs), desc="Chunking query documents"):
        chunks = chunk_text_by_words(
            row["query_text"],
            chunk_words=QUERY_CHUNK_WORDS,
            overlap_words=QUERY_CHUNK_OVERLAP
        )

        for chunk_rank, chunk_text in enumerate(chunks):
            query_chunk_rows.append({
                "query_id": int(row["query_id"]),
                "HADM_ID": int(row["HADM_ID"]),
                "SUBJECT_ID": int(row["SUBJECT_ID"]),
                "split": row["split"],
                "label": int(row["label"]),
                "query_chunk_rank": int(chunk_rank),
                "chunk_text": chunk_text,
                "chunk_text_len": len(chunk_text),
                "chunk_word_len": len(chunk_text.split()),
            })

    query_chunks_df = pd.DataFrame(query_chunk_rows).reset_index(drop=True)
    query_chunks_df["query_chunk_idx"] = query_chunks_df.index.astype(int)

    # Save chunk metadata. By default we keep chunk text too so all retrieval encoder artifacts are reusable.
    bank_to_save = bank_chunks_df.copy()
    query_to_save = query_chunks_df.copy()

    if not SAVE_CHUNK_TEXT_METADATA:
        bank_to_save = bank_to_save.drop(columns=["chunk_text"])
        query_to_save = query_to_save.drop(columns=["chunk_text"])

    save_table(bank_to_save, bank_chunks_path)
    save_table(query_to_save, query_chunks_path)

print("Bank chunks:", bank_chunks_df.shape)
print("Query chunks:", query_chunks_df.shape)
display(bank_chunks_df.head())
display(query_chunks_df.head())


In [ ]:

# ============================================================
# CELL 13: Embed bank/query chunks (with caching)
# - These are the retrieval vectors
# - We save both bank and query chunk embeddings so retrieval
#   does not need to re-encode anything later
# ============================================================

bank_emb_path = RETR_DIR / "bank_chunk_emb.npz"
query_emb_path = RETR_DIR / "query_chunk_emb.npz"

need_rebuild_chunk_emb = (
    FORCE_REBUILD_RETRIEVAL
    or (not bank_emb_path.exists())
    or (not query_emb_path.exists())
)

if not need_rebuild_chunk_emb:
    print("Loading cached retrieval chunk embeddings...")
    bank_chunk_emb = load_npz_array(bank_emb_path)
    query_chunk_emb = load_npz_array(query_emb_path)
else:
    print("Embedding retrieval bank chunks...")
    bank_chunk_emb = embed_texts(
        bank_chunks_df["chunk_text"].tolist(),
        model=encoder,
        batch_size=RETR_BATCH_SIZE,
    )

    print("Embedding retrieval query chunks...")
    query_chunk_emb = embed_texts(
        query_chunks_df["chunk_text"].tolist(),
        model=encoder,
        batch_size=RETR_BATCH_SIZE,
    )

    save_npz_array(bank_emb_path, bank_chunk_emb)
    save_npz_array(query_emb_path, query_chunk_emb)

print("bank_chunk_emb:", bank_chunk_emb.shape)
print("query_chunk_emb:", query_chunk_emb.shape)


In [ ]:

# ============================================================
# CELL 14: Retrieve top chunk hits and collapse to admission-level features
# - Exact torch GPU retrieval
# - Pool query-chunk hits back to UNIQUE bank admissions
# - Exclude same HADM_ID and same SUBJECT_ID for train queries
# - Save wscore_top20 plus optional diagnostics
# ============================================================

retrieval_features_path = RETR_DIR / "retrieval_features.parquet"
top_neigh_map_path = RETR_DIR / "top_unique_hadm_map.pkl"
retrieval_metrics_path = RETR_DIR / "retrieval_feature_metrics.csv"

need_rebuild_features = (
    FORCE_REBUILD_RETRIEVAL
    or (not retrieval_features_path.exists())
    or (not top_neigh_map_path.exists())
)

if not need_rebuild_features:
    print("Loading cached admission-level retrieval features...")
    features_df = pd.read_parquet(retrieval_features_path)
    top_unique_hadm_map = load_pickle(top_neigh_map_path)
else:
    retr_idx, retr_sim = retrieve_topk_torch_gpu(
        query_emb=query_chunk_emb.astype(np.float32),
        bank_emb=bank_chunk_emb.astype(np.float32),
        topk=RETRIEVAL_PER_QUERY_CHUNK,
        block_size=RETR_BLOCK_SIZE,
        use_fp16=True,
    )

    bank_chunk_hadm = bank_chunks_df["HADM_ID"].values.astype(int)
    bank_chunk_subj = bank_chunks_df["SUBJECT_ID"].values.astype(int)
    bank_chunk_label = bank_chunks_df["label"].values.astype(int)

    query_meta = query_docs.set_index("query_id")
    feature_rows = []
    top_unique_hadm_map = {}

    for query_id, grp in tqdm(query_chunks_df.groupby("query_id"), desc="Pooling chunk hits to unique admissions"):
        q = query_meta.loc[int(query_id)]

        best_by_hadm = {}
        query_chunk_row_indices = grp["query_chunk_idx"].values

        for qc_row_idx in query_chunk_row_indices:
            cand_chunk_idxs = retr_idx[qc_row_idx]
            cand_sims = retr_sim[qc_row_idx]

            for bank_chunk_idx, sim in zip(cand_chunk_idxs, cand_sims):
                hadm = int(bank_chunk_hadm[bank_chunk_idx])
                subj = int(bank_chunk_subj[bank_chunk_idx])
                lab = int(bank_chunk_label[bank_chunk_idx])

                # Exclude self-match and same-subject leakage.
                # For val/test this is effectively a no-op because the bank is train-only,
                # but keeping the rule here makes the logic explicit and generic.
                if hadm == int(q["HADM_ID"]):
                    continue
                if subj == int(q["SUBJECT_ID"]):
                    continue

                if (hadm not in best_by_hadm) or (sim > best_by_hadm[hadm]["sim"]):
                    best_by_hadm[hadm] = {
                        "HADM_ID": hadm,
                        "SUBJECT_ID": subj,
                        "label": lab,
                        "sim": float(sim),
                        "best_bank_chunk_idx": int(bank_chunk_idx),
                    }

        unique_neigh = pd.DataFrame(best_by_hadm.values()).sort_values("sim", ascending=False).reset_index(drop=True)

        top_unique_hadm_map[int(query_id)] = unique_neigh["HADM_ID"].head(MAX_FINAL_K).tolist()

        sims_all = unique_neigh["sim"].values.astype(float)
        labs_all = unique_neigh["label"].values.astype(int)

        row = {
            "query_id": int(query_id),
            "HADM_ID": int(q["HADM_ID"]),
            "SUBJECT_ID": int(q["SUBJECT_ID"]),
            "split": q["split"],
            "label": int(q["label"]),
            "max_hosp_day": int(MAX_HOSP_DAY),
            "n_query_chunks": int(len(query_chunk_row_indices)),
            "n_unique_candidates": int(len(unique_neigh)),
        }

        if len(unique_neigh) > 0:
            row["top1_sim"] = float(sims_all[0])
            row["top1_label"] = int(labs_all[0])
            row["mean_top20_sim"] = float(np.mean(sims_all[:min(20, len(sims_all))]))
        else:
            row["top1_sim"] = np.nan
            row["top1_label"] = np.nan
            row["mean_top20_sim"] = np.nan

        for k in TOP_K_LIST:
            k_eff = min(k, len(unique_neigh))

            if k_eff == 0:
                row[f"prop_pos_top{k}"] = np.nan
                row[f"wscore_top{k}"] = np.nan
                row[f"mean_sim_top{k}"] = np.nan
                row[f"max_sim_top{k}"] = np.nan
                row[f"n_eff_top{k}"] = 0
                continue

            sims_k = sims_all[:k_eff]
            labs_k = labs_all[:k_eff]

            row[f"prop_pos_top{k}"] = float(np.mean(labs_k))
            row[f"wscore_top{k}"] = weighted_neighbor_score(sims_k, labs_k)
            row[f"mean_sim_top{k}"] = float(np.mean(sims_k))
            row[f"max_sim_top{k}"] = float(np.max(sims_k))
            row[f"n_eff_top{k}"] = int(k_eff)

        pos_sims = sims_all[labs_all == 1]
        neg_sims = sims_all[labs_all == 0]

        row["best_pos_sim"] = float(pos_sims[0]) if len(pos_sims) else np.nan
        row["best_neg_sim"] = float(neg_sims[0]) if len(neg_sims) else np.nan
        row["pos_neg_gap"] = (
            float(row["best_pos_sim"] - row["best_neg_sim"])
            if (not np.isnan(row["best_pos_sim"])) and (not np.isnan(row["best_neg_sim"]))
            else np.nan
        )

        feature_rows.append(row)

    features_df = pd.DataFrame(feature_rows).sort_values(["split", "HADM_ID"]).reset_index(drop=True)

    save_table(features_df, retrieval_features_path)
    save_pickle(top_unique_hadm_map, top_neigh_map_path)

# Optional quick evaluation of retrieval-only admission feature quality
results = []
score_cols = [f"wscore_top{k}" for k in TOP_K_LIST] + [f"prop_pos_top{k}" for k in TOP_K_LIST] + ["top1_label"]

for split_name in ["train", "val", "test"]:
    sub = features_df[features_df["split"] == split_name].copy()
    if len(sub) == 0:
        continue

    y_true = sub["label"].values.astype(int)

    for score_col in score_cols:
        if score_col not in sub.columns:
            continue
        y_score = sub[score_col].values.astype(float)
        mets = metrics_binary(y_true, y_score)
        results.append({"split": split_name, "score": score_col, **mets})

results_df = pd.DataFrame(results).sort_values(["split", "AUPRC"], ascending=[True, False]).reset_index(drop=True)
results_df.to_csv(retrieval_metrics_path, index=False)

print("features_df:", features_df.shape)
display(features_df.head())
print("\nRetrieval feature metrics (top rows):")
display(results_df.head(20))


In [ ]:

# ============================================================
# CELL 15: Final artifact summary
# - This is the notebook stopping point
# - Notebook 2 starts from the saved files created here
# ============================================================

print("Saved experiment folder:")
print(EXP_DIR)

print("\nMetadata files:")
for p in sorted(META_DIR.glob("*")):
    print(" -", p.name)

print("\nEmbedding files:")
for p in sorted(EMB_DIR.glob("*")):
    print(" -", p.name)

print("\nRetrieval files:")
for p in sorted(RETR_DIR.glob("*")):
    print(" -", p.name)

print("\nNotebook 1 complete.")
print("You can now open Notebook 2 and load this same EXP_DIR without re-encoding.")
